## Modelo de geração de embeddings: BGE-M3

O **BGE-M3** (*BAAI General Embedding Model – M3*), desenvolvido pela Beijing Academy of Artificial Intelligence (BAAI), é um modelo de representação textual baseado na arquitetura Transformer, projetado para gerar embeddings em diferentes idiomas e comprimentos de texto.

O modelo apresenta três características principais:

- **Multilingualidade:** oferece suporte a mais de 100 idiomas, permitindo representar semanticamente textos escritos em diferentes línguas.
- **Multifuncionalidade:** disponibiliza três modalidades de representação textual: *dense retrieval*, *sparse retrieval* e *multi-vector retrieval*.
- **Multigranularidade:** permite processar desde sentenças curtas até documentos extensos, com limite de entrada de até 8.192 tokens.

Neste projeto, é utilizada a modalidade **dense retrieval**, que transforma cada descrição textual em um vetor denso de 1.024 dimensões. Essa representação busca preservar informações semânticas do texto em um espaço vetorial, possibilitando sua utilização como entrada para modelos de aprendizado de máquina e redes neurais.

O BGE-M3 foi escolhido por sua capacidade de processar textos em diferentes idiomas e lidar com descrições extensas, características relevantes para a base de dados do projeto, que contém descrições de vídeos.

**Configurações utilizadas:**
- Modelo: `BAAI/bge-m3`
- Tipo de embedding: *Dense*
- Dimensionalidade dos vetores: 1.024
- Comprimento máximo de entrada: 8.192 tokens
- Precisão: `use_fp16=False`
- Dispositivo de processamento: CPU


**Referência:** CHEN, Jianlv et al. *BGE M3-Embedding: Multi-Lingual, Multi-Functionality, Multi-Granularity Text Embeddings Through Self-Knowledge Distillation*. 2024. Disponível em: https://arxiv.org/abs/2402.03216.

In [ ]:
!pip install -q FlagEmbedding

In [ ]:
import pandas as pd
from FlagEmbedding import BGEM3FlagModel #modelo
from google.colab import drive, userdata
from pathlib import Path
import os
import numpy as np
from tqdm.auto import tqdm

Carregamento dos dados de treino, teste e validação

In [ ]:
drive.mount('/content/drive')
pasta = '/content/drive/MyDrive/SensIA/rede_macro_9classes'

Mounted at /content/drive


In [ ]:
df_treino = pd.read_csv(
    os.path.join(pasta, 'treino.csv')
)

df_validacao = pd.read_csv(
    os.path.join(pasta, 'validacao.csv')
)

df_teste = pd.read_csv(
    os.path.join(pasta, 'teste.csv')
)

In [ ]:
print('Treino:', df_treino.shape)
print('Validação:', df_validacao.shape)
print('Teste:', df_teste.shape)

Treino: (3159, 13)
Validação: (657, 13)
Teste: (684, 13)


Carregamento do modelo

In [ ]:
modelo = BGEM3FlagModel(
    'BAAI/bge-m3',
    use_fp16=False, # utiliza 32 bits para representar cada número
    device='cpu' # especificando, pois o modelo usa GPU
)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Função para gerar os embeddings

In [ ]:
def gerar_embeddings(df, modelo, coluna_texto,
                     batch_size=15):

    textos = df[coluna_texto].fillna('').astype(str).tolist()

    embeddings = []

    for inicio in tqdm( #barra de progresso
        range(0, len(textos), batch_size),
        desc="Gerando embeddings",
        unit="lote"
    ):

        lote = textos[inicio:inicio + batch_size]

        resultado = modelo.encode(
            lote,
            batch_size=batch_size, # Processa n textos por lote
            max_length=8192, #Limite máximo de tokens por texto. Se uma transcrição ultrapassar o limite, o excesso será truncado
            return_dense=True, #Retorna os embeddings densos
            return_sparse=False, #Não gera representações esparsas
            return_colbert_vecs=False #Não gera representações multi-vetoriais
        )

        embeddings.append(resultado['dense_vecs'])

    return np.concatenate(embeddings, axis=0)

Função para salvar os embeddings em um arquivo CSV

In [ ]:
def salvar_embeddings(df, colunas, embeddings, nome_arquivo, pasta_embeddings):

    # Verificar se a quantidade de embeddings corresponde
    # à quantidade de vídeos
    if len(df) != len(embeddings):
        raise ValueError("A quantidade de embeddings não corresponde ao DataFrame.")

    # Selecionar as informações dos vídeos
    df_informacoes = df[
        colunas
    ].reset_index(drop=True)

    # Criar DataFrame com os embeddings
    df_vetores = pd.DataFrame(
        embeddings,
        columns=[f'embedding_{i+1}' for i in range(embeddings.shape[1])]
    )

    # Concatenar informações e embeddings
    df_final = pd.concat(
        [df_informacoes, df_vetores],
        axis=1
    )

    # Caminho completo do arquivo
    caminho = os.path.join(pasta_embeddings, nome_arquivo)

    # Salvar CSV
    df_final.to_csv(
        caminho,
        index=False,
        encoding='utf-8-sig'
    )

    print(f'Arquivo salvo em: {caminho}')
    print(f'Dimensões do arquivo: {df_final.shape}')

    return df_final

## Gerar os embeddings de descricao_global

In [ ]:
embeddings_treino_descricao = gerar_embeddings(df_treino, modelo, 'descricao_global')

Gerando embeddings:   0%|          | 0/211 [00:00<?, ?lote/s]


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 86.80it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  4.91it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 111.08it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.21it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 247.01it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.48it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 295.21it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  4.83it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 226.14it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.48it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 246.14it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  4.28it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 292.86it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  4.33it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 226.52it/s]

Inference Embeddings:

In [ ]:
embeddings_validacao_descricao = gerar_embeddings(df_validacao, modelo, 'descricao_global')

Gerando embeddings:   0%|          | 0/44 [00:00<?, ?lote/s]


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 220.60it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.60it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 176.98it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.86it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 213.90it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.39it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 157.54it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.33it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 134.55it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.89it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 136.36it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.79it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 144.33it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.92it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 205.23it/s]

Inference Embeddings

In [ ]:
embeddings_teste_descricao = gerar_embeddings(df_teste, modelo, 'descricao_global')

Gerando embeddings:   0%|          | 0/46 [00:00<?, ?lote/s]


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 204.09it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.25it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 215.87it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.87it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 235.21it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  4.04it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 237.85it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  4.13it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 250.24it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  4.00it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 318.04it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.06it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 240.57it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 278.01it/s]

Inference Embeddings

Salvando os embeddings em um arquivo CSV.

O arquivo CSV terá 1028 colunas:
- id_video
- descricao_global
- categoria
- split
- 1024 dimensões geradas pelo modelo

In [ ]:
# Pasta para armazenar os arquivos
pasta_embeddings = os.path.join(pasta, 'Embeddings')

os.makedirs(pasta_embeddings, exist_ok=True)

# Treino
df_embeddings_treino_descricao = salvar_embeddings(
    df_treino,
    ['video_id', 'descricao_global', 'categoria', 'split'],
    embeddings_treino_descricao,
    'embeddings_treino_descricao.csv',
    pasta_embeddings
)

# Validação
df_embeddings_validacao_descricao = salvar_embeddings(
    df_validacao,
    ['video_id', 'descricao_global', 'categoria', 'split'],
    embeddings_validacao_descricao,
    'embeddings_validacao_descricao.csv',
    pasta_embeddings
)

# Teste
df_embeddings_teste_descricao = salvar_embeddings(
    df_teste,
    ['video_id', 'descricao_global', 'categoria', 'split'],
    embeddings_teste_descricao,
    'embeddings_teste_descricao.csv',
    pasta_embeddings
)

Arquivo salvo em: /content/drive/MyDrive/SensIA/rede_macro_9classes/Embeddings/embeddings_treino_descricao.csv
Dimensões do arquivo: (3159, 1028)
Arquivo salvo em: /content/drive/MyDrive/SensIA/rede_macro_9classes/Embeddings/embeddings_validacao_descricao.csv
Dimensões do arquivo: (657, 1028)
Arquivo salvo em: /content/drive/MyDrive/SensIA/rede_macro_9classes/Embeddings/embeddings_teste_descricao.csv
Dimensões do arquivo: (684, 1028)


Visualizar os dataframes

In [ ]:
display(df_embeddings_treino_descricao.head())

,video_id,descricao_global,categoria,split,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,...,embedding_1015,embedding_1016,embedding_1017,embedding_1018,embedding_1019,embedding_1020,embedding_1021,embedding_1022,embedding_1023,embedding_1024
0,VIxEMakEHtU,A video series focused on art journaling and p...,Art & Creativity,treino,-0.070557,-0.008093,-0.037610,0.036205,-0.011281,-0.003264,...,0.023857,0.021955,0.026073,-0.022934,0.035242,0.008840,0.031822,-0.019958,-0.026229,-0.035030
1,5xPmCUvmQSY,This video showcases the process of creating a...,Art & Creativity,treino,-0.024065,-0.016170,-0.006906,0.013399,-0.008568,-0.012488,...,-0.025766,0.003320,0.061435,-0.018864,0.017841,0.014002,0.026480,-0.032349,-0.067166,-0.023861
2,TXizGVKDuQc,An art instructional video where Patty teaches...,Art & Creativity,treino,-0.046912,-0.027515,-0.005381,-0.004364,-0.007807,-0.020633,...,-0.025553,0.018176,0.043840,-0.041227,-0.012240,-0.086642,0.033584,-0.028415,0.038091,-0.015158
3,ZkKsQIBEqfU,The host participates in the Book Smartz Proje...,Art & Creativity,treino,-0.049890,-0.015125,0.020362,-0.010704,-0.005552,0.000466,...,-0.029437,0.002733,-0.022896,-0.016655,-0.013837,-0.027565,-0.027150,-0.009502,-0.013173,0.023817
4,g1CVrsLjLDQ,A detailed instructional video guiding viewers...,Art & Creativity,treino,-0.008849,-0.029097,-0.056060,-0.011448,-0.015514,-0.062472,...,0.014471,-0.004657,-0.015181,-0.002294,-0.025377,0.013622,0.050247,-0.053591,-0.006886,-0.030223


In [ ]:
display(df_embeddings_teste_descricao.head())

,video_id,descricao_global,categoria,split,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,...,embedding_1015,embedding_1016,embedding_1017,embedding_1018,embedding_1019,embedding_1020,embedding_1021,embedding_1022,embedding_1023,embedding_1024
0,aXc8aH2BJPk,An instructional video demonstrating how to ac...,Art & Creativity,teste,-0.078521,0.017480,-0.034851,0.027330,-0.026810,-0.019888,...,0.041412,-0.012571,0.031019,-0.023010,-0.020879,-0.010780,0.079500,-0.051387,-0.029268,-0.027299
1,VYccsBM3lXs,This video features Jeff Wall discussing his w...,Art & Creativity,teste,-0.024018,-0.034664,-0.019263,0.045269,-0.008084,0.038837,...,0.019328,-0.036922,-0.003307,-0.018033,0.038581,-0.054574,0.028404,-0.032137,-0.011374,0.023265
2,9QnLwVEXqzs,Review of the Hunt Imperial 0 nib by an artist.,Art & Creativity,teste,-0.040652,-0.007705,-0.045654,0.018957,-0.002572,-0.041731,...,0.013125,-0.020960,0.015432,-0.004480,-0.048836,-0.034065,0.029336,-0.024615,-0.039325,0.016578
3,aDxWwMVkL6M,A series of video scenes featuring Row discuss...,Art & Creativity,teste,-0.017930,-0.013701,0.013770,0.010189,-0.032080,0.022897,...,0.022817,0.004011,0.013050,-0.002258,-0.026406,-0.035772,0.067014,0.018989,0.016876,-0.002655
4,zrfg6nU1tXQ,A video tutorial by Diana on using Gelatos as ...,Art & Creativity,teste,-0.014874,0.043920,-0.017522,-0.000546,-0.024625,-0.024940,...,0.005772,0.007968,0.036214,0.002004,0.018662,-0.004794,0.055704,-0.019522,-0.021824,-0.026689


In [ ]:
display(df_embeddings_validacao_descricao.head())

,video_id,descricao_global,categoria,split,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,...,embedding_1015,embedding_1016,embedding_1017,embedding_1018,embedding_1019,embedding_1020,embedding_1021,embedding_1022,embedding_1023,embedding_1024
0,6RD61KiVdpY,Explore ancient Greek art and its enduring inf...,Art & Creativity,validacao,-0.054318,0.001744,-0.046197,0.018882,-0.020428,-0.043930,...,0.002849,0.025219,0.054957,0.038589,0.036921,-0.025859,0.024806,0.009546,-0.045404,-0.050312
1,143V4wH11Cc,A video interview with Chimamanda Ngozi Adichi...,Art & Creativity,validacao,-0.001391,-0.007771,-0.048227,0.002075,-0.069853,-0.030528,...,0.045871,0.002521,0.022756,-0.022296,0.036385,-0.038106,-0.032704,-0.009045,0.019720,-0.068532
2,aDMLQDqh_AE,A video offering practical advice for artists ...,Art & Creativity,validacao,-0.056628,-0.001845,-0.050763,0.050180,-0.004087,-0.010691,...,0.014351,-0.023775,0.034930,-0.037567,0.024043,-0.012327,0.037121,-0.052681,0.009329,-0.045798
3,dDsevO70nvI,A video featuring Linda MacAulay as she showca...,Art & Creativity,validacao,-0.060629,0.006323,-0.034893,-0.018457,-0.001473,-0.019091,...,-0.002506,-0.013534,0.010160,-0.051278,0.014098,-0.011740,0.001868,-0.023638,-0.023390,-0.051082
4,XuFBpgwZLF0,An instructional video on creating a coral ree...,Art & Creativity,validacao,-0.023191,0.030929,-0.003113,-0.002039,-0.035104,-0.037456,...,-0.010688,-0.003892,-0.030207,-0.015816,-0.025287,0.018700,-0.008682,0.004976,0.011936,-0.028221
